# Job Market Intelligence – Data Collection

This notebook runs the No Fluff Jobs data-collection pipeline.

Scraping logic is implemented in the `src/nfj` package. Raw datasets are stored in `data/raw/` and excluded from version control. All network operations are disabled by default and must be enabled explicitly with the corresponding `RUN_*` flag.


In [1]:
import logging
from pathlib import Path
import sys

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger(__name__)

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from nfj import (
    collect_job_urls,
    refresh_job_records,
    refresh_missing_salary_periods,
    scrape_jobs,
)
from nfj.config import DEFAULT_DELAY, ERROR_PATH, OUTPUT_PATH, URLS_PATH

JOBS_PATH = OUTPUT_PATH
ERRORS_PATH = ERROR_PATH

SCRAPE_DELAY = max(DEFAULT_DELAY, 3.0)
HEADLESS = False

logger.info("Project root: %s", PROJECT_ROOT)
logger.info("URL file: %s", URLS_PATH)
logger.info("Job file: %s", JOBS_PATH)
logger.info("Error file: %s", ERRORS_PATH)


2026-08-28 20:05:16,505 | INFO | Project root: G:\pandas\job_market_intelligence
2026-08-28 20:05:16,507 | INFO | URL file: G:\pandas\job_market_intelligence\data\raw\nofluff_it_urls.csv
2026-08-28 20:05:16,507 | INFO | Job file: G:\pandas\job_market_intelligence\data\raw\nofluff_it_jobs.csv
2026-08-28 20:05:16,507 | INFO | Error file: G:\pandas\job_market_intelligence\data\raw\nofluff_scraping_errors.csv


## 1. Collect job URLs

Collect the current set of unique job-offer URLs from all configured No Fluff Jobs categories.


In [2]:
RUN_URL_COLLECTION = False

if RUN_URL_COLLECTION:
    df_urls = collect_job_urls()
else:
    logger.info("URL collection skipped.")


2026-08-28 20:05:16,521 | INFO | URL collection skipped.


## 2. Scrape job offers

Scrape offers from the URL list and save them to the active raw dataset. Existing records are retained, while URLs that have not been collected are scraped.


In [3]:
RUN_SCRAPING = False

if RUN_SCRAPING:
    df_jobs = scrape_jobs(
        urls_path=URLS_PATH,
        output_path=JOBS_PATH,
        error_path=ERRORS_PATH,
        delay=SCRAPE_DELAY,
        headless=HEADLESS,
        max_consecutive_page_errors=5,
    )
else:
    logger.info("Job scraping skipped.")


2026-08-28 20:05:16,553 | INFO | Job scraping skipped.


## 3. Refresh missing salary periods

Re-scrape paid offers whose `salary_period` is missing. Existing salary values are preserved when the period still cannot be identified.


In [4]:
RUN_SALARY_PERIOD_REFRESH = False

if RUN_SALARY_PERIOD_REFRESH:
    df_jobs = refresh_missing_salary_periods(
        output_path=JOBS_PATH,
        error_path=ERRORS_PATH,
        delay=SCRAPE_DELAY,
        headless=HEADLESS,
    )
else:
    logger.info("Salary-period refresh skipped.")


2026-08-28 20:05:16,571 | INFO | Salary-period refresh skipped.


## 4. Refresh selected offers

Use this section only when specific records need to be scraped again after a parser correction. Add exact URLs to `URLS_TO_REFRESH` and enable the flag.


In [5]:
RUN_SELECTED_REFRESH = False

URLS_TO_REFRESH = []

if RUN_SELECTED_REFRESH:
    if not URLS_TO_REFRESH:
        raise ValueError(
            "Add at least one URL to URLS_TO_REFRESH before enabling the refresh."
        )

    df_jobs = refresh_job_records(
        urls=URLS_TO_REFRESH,
        output_path=JOBS_PATH,
        error_path=ERRORS_PATH,
        delay=SCRAPE_DELAY,
        headless=HEADLESS,
    )
else:
    logger.info("Selected-offer refresh skipped.")


2026-08-28 20:05:16,592 | INFO | Selected-offer refresh skipped.


## 5. Validate collected data

Check file availability, schema completeness, URL uniqueness, invalid pages, and fields affected by parser corrections. This section is local and safe to run with all network-operation flags disabled.


In [6]:
required_files = [URLS_PATH, JOBS_PATH]
missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    missing_list = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Required raw-data files are missing:\n"
        f"{missing_list}\n"
        "Run the relevant collection step before validation."
    )

df_urls = pd.read_csv(URLS_PATH)
df_jobs = pd.read_csv(JOBS_PATH)

required_url_columns = {"url"}
required_job_columns = {
    "job_id",
    "url",
    "title",
    "category",
    "salary_min",
    "salary_max",
    "salary_currency",
    "salary_period",
    "required_skills",
    "responsibilities",
}

missing_url_columns = required_url_columns - set(df_urls.columns)
missing_job_columns = required_job_columns - set(df_jobs.columns)

if missing_url_columns or missing_job_columns:
    raise ValueError(
        "Missing required columns. "
        f"URL dataset: {sorted(missing_url_columns)}; "
        f"job dataset: {sorted(missing_job_columns)}"
    )

invalid_page_markers = (
    "Ta strona nie działa|"
    "Oferta pracy wygasła|"
    "Oferta wygasła"
)

invalid_job_mask = (
    df_jobs["title"]
    .fillna("")
    .str.contains(
        invalid_page_markers,
        case=False,
        regex=True,
    )
)

required_skills_leak = (
    df_jobs["required_skills"]
    .fillna("")
    .str.contains("Opis oferty", case=False, regex=False)
)

responsibilities_leak = (
    df_jobs["responsibilities"]
    .fillna("")
    .str.contains("Opis oferty", case=False, regex=False)
)

salary_without_period = (
    df_jobs[["salary_min", "salary_max", "salary_currency"]]
    .notna()
    .any(axis=1)
    & df_jobs["salary_period"].isna()
)

missing_urls = set(df_urls["url"].dropna()) - set(df_jobs["url"].dropna())

validation_summary = pd.DataFrame(
    {
        "metric": [
            "Collected URLs",
            "Unique collected URLs",
            "Job records",
            "Unique job URLs",
            "Duplicate job URLs",
            "Missing job IDs",
            "Duplicate job IDs",
            "URLs without a job record",
            "Invalid job pages",
            "Salary values without period",
            "Required-skills section leaks",
            "Responsibilities section leaks",
            "Missing categories",
        ],
        "value": [
            len(df_urls),
            df_urls["url"].nunique(),
            len(df_jobs),
            df_jobs["url"].nunique(),
            int(df_jobs["url"].duplicated().sum()),
            int(df_jobs["job_id"].isna().sum()),
            int(df_jobs["job_id"].duplicated().sum()),
            len(missing_urls),
            int(invalid_job_mask.sum()),
            int(salary_without_period.sum()),
            int(required_skills_leak.sum()),
            int(responsibilities_leak.sum()),
            int(df_jobs["category"].isna().sum()),
        ],
    }
)

validation_summary


,metric,value
0,Collected URLs,3360
1,Unique collected URLs,3360
2,Job records,3360
3,Unique job URLs,3360
4,Duplicate job URLs,0
5,Missing job IDs,0
6,Duplicate job IDs,0
7,URLs without a job record,0
8,Invalid job pages,0
9,Salary values without period,1


### Validation criteria

A successful collection should contain no duplicated URLs or job IDs, invalid-page records, parser section leaks, or missing categories. URLs without a job record may occur when an offer expires or becomes unavailable between URL collection and scraping. Salary values without an identified period require a targeted refresh or should be excluded during salary preparation.
